# AQI analysis — Patia, Bhubaneswar (CPCB hourly, Nov 2023 – Dec 2025)

Clean, top-to-bottom version of the notebook. **Run with `Kernel → Restart & Run All`.**

| Section | What it does |
|---|---|
| 1 | Config + imports |
| 2 | Load the 3 CPCB Excel files onto a full hourly grid |
| 3 | Quality control (negatives, stuck sensors, stuck-at-minimum) |
| 4 | ERA5 weather from Open-Meteo (cached to CSV after first download) |
| 5 | Features (weather + weather history + calendar, **no pollutants, no day-of-year**) |
| 6 | CPCB AQI functions |
| 7 | **Goal B** – weather-only model per pollutant (chronological split) |
| 8 | Goal B → daily AQI category + season breakdown |
| 9 | **Goal A** – day-ahead forecast vs persistence |
| 10 | **Experiment 1** – random vs chronological split for *our* models |
| 11 | **Experiment 2** – literature setups (same-day pollutants → AQI) vs ours, both splits |
| 12 | Save all result tables |

In [1]:
# ===================== 1. CONFIG + IMPORTS =====================
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import os
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

FILES = ["Bhubneshwar-1h-data-2023.xlsx",
         "Bhubhneshwar-1h-data-2024.xlsx",
         "Bhubaneswar-1h-data-2025.xlsx"]
SKIPROWS   = 10
LAT, LON   = 20.35, 85.82                 # Patia, Bhubaneswar
ERA5_CACHE = "era5_patia.csv"             # downloaded once, then reused
RESULTS    = "results"                    # output folder for tables

SPLIT_DATE = "2025-01-01"                 # train < SPLIT_DATE
TEST_END   = "2026-01-01"                 # test  = [SPLIT_DATE, TEST_END)

TARGETS  = ["PM2.5", "PM10", "NO2", "SO2", "CO", "OZONE", "NH3"]
AQI_POLS = ["PM2.5", "PM10", "NO2", "SO2", "OZONE", "NH3"]   # CO excluded (faulty sensor, see QC)
SEASON = {12: "Winter", 1: "Winter", 2: "Winter",
          3: "Summer", 4: "Summer", 5: "Summer",
          6: "Monsoon", 7: "Monsoon", 8: "Monsoon", 9: "Monsoon",
          10: "Post-Monsoon", 11: "Post-Monsoon"}

def rf(seed=42):
    # regularised RF: leaves of >=50 hours stop the model memorising timestamps
    return RandomForestRegressor(n_estimators=300, min_samples_leaf=50, max_features=0.5,
                                 max_depth=12, n_jobs=-1, random_state=seed)

os.makedirs(RESULTS, exist_ok=True)

In [2]:
# ===================== 2. LOAD CPCB DATA =====================
def load_cpcb(files, skiprows=SKIPROWS):
    parts = []
    for f in files:
        d = pd.read_excel(f, skiprows=skiprows)
        print(f"{f}: {d.shape}")
        parts.append(d)
    df = pd.concat(parts, ignore_index=True)
    df["From Date"] = pd.to_datetime(df["From Date"], dayfirst=True, errors="coerce")
    df = df.dropna(subset=["From Date"])                       # drops footer / summary rows
    df = df.drop_duplicates(subset="From Date").sort_values("From Date").set_index("From Date")
    df = df.drop(columns=["To Date", "BP", "XYLENE", "MP-Xylene"], errors="ignore")
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")          # "None"/text -> NaN
    full = pd.date_range(df.index.min(), df.index.max(), freq="h")
    df = df.reindex(full)                                      # full hourly grid
    df.index.name = "time"
    return df

raw = load_cpcb(FILES)
print("\nGrid:", raw.index.min(), "→", raw.index.max(), "| rows:", len(raw))
print(raw.isna().mean().round(2).sort_values().to_string())

Bhubneshwar-1h-data-2023.xlsx: (1564, 21)
Bhubhneshwar-1h-data-2024.xlsx: (8669, 21)
Bhubaneswar-1h-data-2025.xlsx: (8596, 21)

Grid: 2023-10-27 16:00:00 → 2025-12-31 23:00:00 | rows: 19112
OZONE      0.02
BENZENE    0.02
TOLUENE    0.02
RH         0.02
SR         0.02
AT         0.02
RF         0.02
PM10       0.03
NO         0.03
NOX        0.03
SO2        0.03
CO         0.04
WD         0.04
PM2.5      0.05
NO2        0.07
NH3        0.07


In [3]:
# ===================== 3. QUALITY CONTROL =====================
def remove_flatlines(s, n=6):
    # NaN where the same value repeats for n+ consecutive hours (stuck sensor)
    grp = (s != s.shift()).cumsum()
    run = s.groupby(grp).transform("size")
    return s.mask((run >= n) & s.notna())

def clean(raw):
    d = raw.copy()
    report = {}
    before = d[TARGETS].notna().sum()
    d[TARGETS] = d[TARGETS].where(d[TARGETS] >= 0)
    report["negative"] = before - d[TARGETS].notna().sum()

    before = d[TARGETS].notna().sum()
    for t in TARGETS:
        d[t] = remove_flatlines(d[t])
    report["flatline_6h"] = before - d[TARGETS].notna().sum()

    before = d[TARGETS].notna().sum()
    d[["NO2", "NH3"]] = d[["NO2", "NH3"]].mask(d[["NO2", "NH3"]] <= 0.1)
    report["stuck_at_0.1"] = before - d[TARGETS].notna().sum()

    report = pd.DataFrame(report).fillna(0).astype(int)
    report["missing_%_after"] = (d[TARGETS].isna().mean() * 100).round(1)
    return d, report

data, qc_report = clean(raw)
print("Rows removed by each QC rule:\n", qc_report, "\n")
print("Monthly medians (look for jumps / stuck months):")
print(data[["NO2", "CO", "NH3"]].resample("MS").median().round(2).to_string())

Rows removed by each QC rule:
        negative  flatline_6h  stuck_at_0.1  missing_%_after
PM2.5         0            0             0              4.7
PM10          0            7             0              3.4
NO2           0            0            91              7.5
SO2           0            0             0              3.0
CO            0          136             0              4.4
OZONE         0            0             0              1.9
NH3           0           51           347              9.0 

Monthly medians (look for jumps / stuck months):
              NO2    CO    NH3
time                          
2023-10-01  16.00  0.70  19.15
2023-11-01  13.52  0.66  18.40
2023-12-01  19.28  0.72  19.38
2024-01-01  20.98  0.88  19.15
2024-02-01  13.95  1.33  15.43
2024-03-01   7.76  1.65  10.55
2024-04-01   0.20  1.72   0.15
2024-05-01   3.58  0.47   0.20
2024-06-01   8.46  0.47   5.43
2024-07-01   6.20  0.38   6.25
2024-08-01   4.90  0.37   5.00
2024-09-01   5.60  0.38   3.98
2024

In [4]:
# ===================== 4. ERA5 WEATHER (Open-Meteo) =====================
def get_era5(start, end, cache=ERA5_CACHE):
    if os.path.exists(cache):
        met = pd.read_csv(cache, index_col=0, parse_dates=True)
        print("ERA5 loaded from cache:", cache)
        return met
    import requests
    params = {"latitude": LAT, "longitude": LON, "start_date": start, "end_date": end,
              "hourly": "wind_speed_10m,boundary_layer_height,surface_pressure,cloud_cover",
              "models": "era5", "timezone": "Asia/Kolkata", "wind_speed_unit": "ms"}
    r = requests.get("https://archive-api.open-meteo.com/v1/archive", params=params, timeout=120)
    r.raise_for_status()
    met = pd.DataFrame(r.json()["hourly"])
    met["time"] = pd.to_datetime(met["time"])
    met = met.set_index("time")
    met.to_csv(cache)
    print("ERA5 downloaded and cached:", cache)
    return met

met = get_era5(str(data.index.min().date()), str(data.index.max().date()))
data = data.drop(columns=met.columns, errors="ignore").join(met, how="left")
print(met.isna().mean().round(2).to_string())

ERA5 downloaded and cached: era5_patia.csv
wind_speed_10m           0.00
boundary_layer_height    0.23
surface_pressure         0.00
cloud_cover              0.00


In [5]:
# ===================== 5. FEATURES (weather + time only) =====================
def build_features(d):
    d = d.copy()
    idx = d.index
    d["hour_sin"], d["hour_cos"] = np.sin(2*np.pi*idx.hour/24), np.cos(2*np.pi*idx.hour/24)
    d["dow_sin"],  d["dow_cos"]  = np.sin(2*np.pi*idx.dayofweek/7), np.cos(2*np.pi*idx.dayofweek/7)
    d["WD_sin"],   d["WD_cos"]   = np.sin(np.deg2rad(d["WD"])), np.cos(np.deg2rad(d["WD"]))
    d["season"] = idx.month.map(SEASON)

    # weather history (last 24–72 h) — still weather only, no leakage
    for c in ["AT", "RH", "SR"]:
        d[f"{c}_mean24"] = d[c].rolling("24h", min_periods=12).mean()
    d["AT_range24"] = d["AT"].rolling("24h", min_periods=12).max() - d["AT"].rolling("24h", min_periods=12).min()
    d["RH_mean72"]  = d["RH"].rolling("72h", min_periods=24).mean()
    d["RF_sum24"]   = d["RF"].rolling("24h", min_periods=1).sum()
    d["RF_sum72"]   = d["RF"].rolling("72h", min_periods=1).sum()
    d["WD_sin24"]   = d["WD_sin"].rolling("24h", min_periods=12).mean()
    d["WD_cos24"]   = d["WD_cos"].rolling("24h", min_periods=12).mean()

    era5 = [c for c in ["wind_speed_10m", "boundary_layer_height", "surface_pressure", "cloud_cover"] if c in d]
    if "wind_speed_10m" in d:
        d["WS_mean24"] = d["wind_speed_10m"].rolling("24h", min_periods=12).mean(); era5.append("WS_mean24")
    if "boundary_layer_height" in d:
        d["BLH_mean24"] = d["boundary_layer_height"].rolling("24h", min_periods=12).mean(); era5.append("BLH_mean24")
        if "wind_speed_10m" in d:
            d["ventilation"] = d["wind_speed_10m"] * d["boundary_layer_height"]; era5.append("ventilation")

    cols = dict(
        num  = ["AT", "RH", "SR", "WD_sin", "WD_cos", "AT_mean24", "RH_mean24", "SR_mean24",
                "AT_range24", "RH_mean72", "WD_sin24", "WD_cos24"] + era5,
        rain = ["RF", "RF_sum24", "RF_sum72"],
        cyc  = ["hour_sin", "hour_cos", "dow_sin", "dow_cos"],      # NO day-of-year (caused memorisation)
        cat  = ["season"],
    )
    return d, cols

def add_pollution_lags(d):
    # day-ahead inputs: only values >= 24 h old (requires the full hourly grid from section 2)
    d = d.copy(); lag_cols = []
    for p in TARGETS:
        d[f"{p}_lag24"]   = d[p].shift(24)
        d[f"{p}_lag48"]   = d[p].shift(48)
        d[f"{p}_prevday"] = d[p].shift(24).rolling(24, min_periods=12).mean()
        lag_cols += [f"{p}_lag24", f"{p}_lag48", f"{p}_prevday"]
    return d, lag_cols

def make_model(cols, seed=42, log_target=True):
    pre = ColumnTransformer([
        ("num",  make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), cols["num"]),
        ("rain", make_pipeline(SimpleImputer(strategy="constant", fill_value=0),
                               FunctionTransformer(np.log1p)), cols["rain"]),
        ("cyc",  "passthrough", cols["cyc"]),
        ("cat",  make_pipeline(SimpleImputer(strategy="most_frequent"),
                               OneHotEncoder(handle_unknown="ignore")), cols["cat"]),
    ])
    pipe = Pipeline([("pre", pre), ("model", rf(seed))])
    return TransformedTargetRegressor(regressor=pipe, func=np.log1p, inverse_func=np.expm1) if log_target else pipe

def feats(cols):
    return cols["num"] + cols["rain"] + cols["cyc"] + cols["cat"]

data, COLS_B = build_features(data)
data, LAG_COLS = add_pollution_lags(data)
COLS_A = {**COLS_B, "num": COLS_B["num"] + LAG_COLS}
FEAT_B, FEAT_A = feats(COLS_B), feats(COLS_A)
print("Goal B features:", len(FEAT_B), "| Goal A features:", len(FEAT_A))

Goal B features: 27 | Goal A features: 48


In [6]:
# ===================== 6. CPCB AQI FUNCTIONS =====================
I_LEVELS = [0, 50, 100, 200, 300, 400, 500]
C_BP = {   # CPCB National AQI breakpoints (CO in mg/m3, others ug/m3)
    "PM2.5": [0, 30, 60, 90, 120, 250, 380],
    "PM10":  [0, 50, 100, 250, 350, 430, 510],
    "NO2":   [0, 40, 80, 180, 280, 400, 520],
    "OZONE": [0, 50, 100, 168, 208, 748, 1000],
    "CO":    [0, 1, 2, 10, 17, 34, 51],
    "SO2":   [0, 40, 80, 380, 800, 1600, 2400],
    "NH3":   [0, 200, 400, 800, 1200, 1800, 2400],
}
LABELS = ["Good", "Satisfactory", "Moderate", "Poor", "Very Poor", "Severe"]

def daily_conc(h):
    # CPCB averaging: 24-h mean (>=16 valid hours) for most; max 8-h mean for CO and O3
    d = pd.DataFrame(index=h.resample("D").mean().index)
    for p in ["PM2.5", "PM10", "NO2", "SO2", "NH3"]:
        g = h[p].resample("D")
        d[p] = g.mean().where(g.count() >= 16)
    for p in ["CO", "OZONE"]:
        d[p] = h[p].rolling("8h", min_periods=6).mean().resample("D").max()
    return d

def compute_aqi(d, pols=AQI_POLS):
    si = pd.DataFrame({p: np.interp(d[p], C_BP[p], I_LEVELS) for p in pols}, index=d.index)
    si = si.where(d[pols].notna())
    valid = (si.notna().sum(axis=1) >= 3) & si[["PM2.5", "PM10"]].notna().any(axis=1)
    return si.max(axis=1).where(valid), si.fillna(-1).idxmax(axis=1).where(valid)

def aqi_category(a):
    return pd.cut(pd.Series(a), bins=[-1, 50, 100, 200, 300, 400, np.inf], labels=LABELS)

def aqi_scores(a_true, a_pred):
    ok = a_true.notna() & a_pred.notna()
    ct, cp = aqi_category(a_true[ok]).cat.codes, aqi_category(a_pred[ok]).cat.codes
    return {"days": int(ok.sum()),
            "exact_acc": (ct == cp).mean(),
            "within_1": (abs(ct - cp) <= 1).mean(),
            "AQI_MAE": (a_true[ok] - a_pred[ok]).abs().mean()}

In [7]:
# ===================== 7. GOAL B: WEATHER-ONLY MODEL (chronological) =====================
train = data[data.index < SPLIT_DATE]
test  = data[(data.index >= SPLIT_DATE) & (data.index < TEST_END)]
print("Train:", train.index.min(), "→", train.index.max(), "| Test:", test.index.min(), "→", test.index.max())

def fit_predict(train, test, cols, features, seed=42):
    preds, models = pd.DataFrame(index=test.index, columns=TARGETS, dtype=float), {}
    for t in TARGETS:
        tr = train.dropna(subset=[t])                       # each pollutant uses its own rows
        m = make_model(cols, seed)
        m.fit(tr[features], tr[t])
        preds[t], models[t] = m.predict(test[features]), m
    return preds, models

def pollutant_table(train, test, preds, models, features):
    rows = []
    for t in TARGETS:
        tr = train.dropna(subset=[t])
        clim = tr[t].groupby(tr.index.month * 100 + tr.index.hour).mean()      # month x hour baseline
        base = pd.Series(clim.reindex(test.index.month * 100 + test.index.hour).values, index=test.index)
        m  = test[t].notna() & base.notna()
        dt, dp = test[t].resample("D").mean(), preds[t].where(test[t].notna()).resample("D").mean()
        dm = dt.notna() & dp.notna()
        mae, bmae = mean_absolute_error(test.loc[m, t], preds.loc[m, t]), mean_absolute_error(test.loc[m, t], base[m])
        rows.append({"pollutant": t,
                     "train_R2": r2_score(tr[t], models[t].predict(tr[features])),
                     "test_R2": r2_score(test.loc[m, t], preds.loc[m, t]),
                     "baseline_R2": r2_score(test.loc[m, t], base[m]),
                     "daily_R2": r2_score(dt[dm], dp[dm]),
                     "MAE": mae, "skill_vs_baseline": 1 - mae / bmae,
                     "bias": (preds.loc[m, t] - test.loc[m, t]).mean()})
    return pd.DataFrame(rows).set_index("pollutant")

preds_B, models_B = fit_predict(train, test, COLS_B, FEAT_B)
tab_B = pollutant_table(train, test, preds_B, models_B, FEAT_B)
print(tab_B.round(2))

Train: 2023-10-27 16:00:00 → 2024-12-31 23:00:00 | Test: 2025-01-01 00:00:00 → 2025-12-31 23:00:00
           train_R2  test_R2  baseline_R2  daily_R2    MAE  skill_vs_baseline  \
pollutant                                                                       
PM2.5          0.71     0.69         0.59      0.80  14.13               0.22   
PM10           0.63     0.55         0.47      0.68  30.14               0.19   
NO2            0.75     0.37         0.31      0.24   7.98               0.06   
SO2            0.47     0.21        -0.13      0.16   3.48               0.31   
CO             0.53    -0.15        -0.93     -0.37   0.53               0.27   
OZONE          0.83     0.74         0.47      0.68  13.65               0.32   
NH3            0.64    -0.33        -0.39     -0.38  11.31               0.06   

            bias  
pollutant         
PM2.5      -3.40  
PM10       -7.90  
NO2        -6.39  
SO2        -1.08  
CO         -0.09  
OZONE      -2.82  
NH3       -10.25  


In [8]:
# ===================== 8. GOAL B → DAILY AQI + SEASONS =====================
a_true, dom_true = compute_aqi(daily_conc(test[TARGETS]))
a_B,    dom_B    = compute_aqi(daily_conc(preds_B.where(test[TARGETS].notna())))
ok = a_true.notna() & a_B.notna()

print("AQI (weather-only, no CO):", {k: round(v, 3) for k, v in aqi_scores(a_true, a_B).items()})
print("Dominant-pollutant match:", round((dom_true[ok] == dom_B[ok]).mean(), 3))
print(pd.crosstab(aqi_category(a_true[ok]), aqi_category(a_B[ok]), rownames=["Actual"], colnames=["Predicted"]))

s = pd.DataFrame({"season": a_true[ok].index.month.map(SEASON),
                  "hit": (aqi_category(a_true[ok]).cat.codes == aqi_category(a_B[ok]).cat.codes).values,
                  "abs_err": (a_true[ok] - a_B[ok]).abs().values,
                  "dominant": dom_true[ok].values})
season_tab = s.groupby("season").agg(days=("hit", "size"), exact_acc=("hit", "mean"), AQI_MAE=("abs_err", "mean"))
print("\n", season_tab.round(2))
print("\n", pd.crosstab(s["season"], s["dominant"]))

AQI (weather-only, no CO): {'days': 352, 'exact_acc': 0.653, 'within_1': 0.989, 'AQI_MAE': 24.198}
Dominant-pollutant match: 0.574
Predicted     Good  Satisfactory  Moderate  Poor
Actual                                          
Good            93            14         0     0
Satisfactory    44            63         5     0
Moderate         1            31        42     6
Poor             0             1        12    32
Very Poor        0             0         2     6

               days  exact_acc  AQI_MAE
season                                
Monsoon        115       0.71    11.31
Post-Monsoon    58       0.50    41.11
Summer          89       0.61    20.83
Winter          90       0.72    33.09

 dominant      NO2  OZONE  PM10  PM2.5
season                               
Monsoon         3     40    71      1
Post-Monsoon    0      1    30     27
Summer          0     68    21      0
Winter          0     27    14     49


In [9]:
# ===================== 9. GOAL A: DAY-AHEAD FORECAST vs PERSISTENCE =====================
preds_A, models_A = fit_predict(train, test, COLS_A, FEAT_A)
tab_A = pollutant_table(train, test, preds_A, models_A, FEAT_A)

cmp = pd.DataFrame({"test_R2_B(weather)": tab_B["test_R2"], "test_R2_A(day-ahead)": tab_A["test_R2"],
                    "daily_R2_B": tab_B["daily_R2"], "daily_R2_A": tab_A["daily_R2"]})
print(cmp.round(2))

a_A, _ = compute_aqi(daily_conc(preds_A.where(test[TARGETS].notna())))
a_pers = compute_aqi(daily_conc(data[TARGETS]))[0].shift(1).reindex(a_true.index)   # tomorrow = today
common = a_true.notna() & a_B.notna() & a_A.notna() & a_pers.notna()
aqi_cmp = pd.DataFrame({"Persistence (today=tomorrow)": aqi_scores(a_true[common], a_pers[common]),
                        "Goal B: weather only":         aqi_scores(a_true[common], a_B[common]),
                        "Goal A: day-ahead forecast":   aqi_scores(a_true[common], a_A[common])}).T
print("\n", aqi_cmp.round(3))

           test_R2_B(weather)  test_R2_A(day-ahead)  daily_R2_B  daily_R2_A
pollutant                                                                  
PM2.5                    0.69                  0.77        0.80        0.88
PM10                     0.55                  0.69        0.68        0.84
NO2                      0.37                  0.51        0.24        0.52
SO2                      0.21                  0.32        0.16        0.35
CO                      -0.15                  0.37       -0.37        0.50
OZONE                    0.74                  0.79        0.68        0.77
NH3                     -0.33                  0.07       -0.38        0.09

                                days  exact_acc  within_1  AQI_MAE
Persistence (today=tomorrow)  344.0      0.692     0.997   21.441
Goal B: weather only          344.0      0.660     0.991   23.926
Goal A: day-ahead forecast    344.0      0.741     0.997   18.700


## 10. Experiment 1 — random vs chronological split for **our** models

Same model, same features; only the split changes.
* **random**: 80/20 random hours over the whole period (5 seeds → mean ± sd)
* **chronological**: train < 2025, test = 2025

If random ≫ chronological, a random split inflates the score (neighbouring hours/days leak between train and test).

In [10]:
# ===================== 10. EXPERIMENT 1: RANDOM vs CHRONOLOGICAL (hourly pollutant models) =====================
N_SEEDS = 5
period = data[data.index < TEST_END]

def split_rows(index, kind, seed):
    if kind == "chronological":
        return np.asarray(index < SPLIT_DATE), np.asarray((index >= SPLIT_DATE) & (index < TEST_END))
    m = np.random.RandomState(seed).rand(len(index)) < 0.8
    return m, ~m

exp1 = []
for setup, cols, features in [("B. weather only", COLS_B, FEAT_B), ("A. day-ahead", COLS_A, FEAT_A)]:
    for kind in ["random", "chronological"]:
        for seed in (range(N_SEEDS) if kind == "random" else [0]):
            tr_m, te_m = split_rows(period.index, kind, seed)
            tr, te = period[tr_m], period[te_m]
            for t in TARGETS:
                trt = tr.dropna(subset=[t]); tet = te.dropna(subset=[t])
                m = make_model(cols, seed); m.fit(trt[features], trt[t])
                p = m.predict(tet[features])
                exp1.append({"setup": setup, "split": kind, "seed": seed, "pollutant": t,
                             "R2": r2_score(tet[t], p), "MAE": mean_absolute_error(tet[t], p)})
        print("done:", setup)

exp1 = pd.DataFrame(exp1)
exp1_summary = exp1.groupby(["setup", "pollutant", "split"])["R2"].agg(["mean", "std"]).unstack("split").round(3)
print(exp1_summary)

r2_wide = exp1.groupby(["setup", "pollutant", "split"])["R2"].mean().unstack("split")
r2_wide["inflation (random - chrono)"] = r2_wide["random"] - r2_wide["chronological"]
print("\nR2 inflation from random split:\n", r2_wide.round(3))

done: B. weather only
done: B. weather only
done: A. day-ahead
done: A. day-ahead
                                   mean                  std       
split                     chronological random chronological random
setup           pollutant                                          
A. day-ahead    CO                0.374  0.487           NaN  0.025
                NH3               0.086  0.781           NaN  0.022
                NO2               0.504  0.700           NaN  0.028
                OZONE             0.784  0.822           NaN  0.006
                PM10              0.691  0.686           NaN  0.018
                PM2.5             0.776  0.761           NaN  0.019
                SO2               0.322  0.404           NaN  0.043
B. weather only CO               -0.121  0.444           NaN  0.014
                NH3              -0.326  0.503           NaN  0.028
                NO2               0.373  0.687           NaN  0.020
                OZONE             

## 11. Experiment 2 — literature setups vs ours (daily AQI)

| Setup | Inputs | Mimics |
|---|---|---|
| A. same-day pollutants | that day's PM2.5, PM10, NO2, SO2, O3, NH3, NO, NOx | Pande 2024, Gupta 2023, Mubeen 2025 |
| B. weather only | that day's weather + calendar | our Goal B |
| C. day-ahead | today's weather + **yesterday's** pollutants & AQI | our Goal A |
| Persistence | AQI tomorrow = AQI today | baseline |

All predict **daily CPCB AQI directly**, scored on the same days, with both split types.

In [11]:
# ===================== 11. EXPERIMENT 2: LITERATURE SETUPS × SPLIT (daily AQI) =====================
dd   = data[data.index < TEST_END]
conc = daily_conc(dd[TARGETS])
aqi, _ = compute_aqi(conc)
pers = aqi.shift(1)

A = conc[AQI_POLS].copy()
for c in ["NO", "NOX"]:
    if c in dd:
        A[c] = dd[c].resample("D").mean()

w_cols = [c for c in ["AT", "RH", "SR", "WD_sin", "WD_cos", "wind_speed_10m",
                      "boundary_layer_height", "surface_pressure", "cloud_cover", "ventilation"] if c in dd]
B = dd[w_cols].resample("D").mean()
B["RF_sum"]   = dd["RF"].resample("D").sum(min_count=1)
B["AT_range"] = dd["AT"].resample("D").max() - dd["AT"].resample("D").min()
B["dow_sin"], B["dow_cos"]     = np.sin(2*np.pi*B.index.dayofweek/7), np.cos(2*np.pi*B.index.dayofweek/7)
B["month_sin"], B["month_cos"] = np.sin(2*np.pi*B.index.month/12),   np.cos(2*np.pi*B.index.month/12)

lag = conc[AQI_POLS].shift(1).add_suffix("_lag1")
lag["AQI_lag1"] = aqi.shift(1)
C = pd.concat([B, lag], axis=1)

SETS = {"A. same-day pollutants (literature)": A, "B. weather only": B, "C. day-ahead forecast": C}
days = aqi.index[aqi.notna() & pers.notna()]

def daily_split(index, kind, seed):
    if kind == "chronological":
        return np.asarray(index < SPLIT_DATE), np.asarray((index >= SPLIT_DATE) & (index < TEST_END))
    m = np.random.RandomState(seed).rand(len(index)) < 0.8
    return m, ~m

def score_daily(y, p, y_pers):
    p = pd.Series(np.asarray(p), index=y.index)
    ct, cp = aqi_category(y).cat.codes.values, aqi_category(p).cat.codes.values
    return {"R2": r2_score(y, p), "MAE": mean_absolute_error(y, p),
            "exact_acc": (ct == cp).mean(), "within_1": (abs(ct - cp) <= 1).mean(),
            "skill_vs_pers": 1 - mean_squared_error(y, p) / mean_squared_error(y, y_pers)}

exp2 = []
for kind in ["random", "chronological"]:
    for seed in (range(N_SEEDS) if kind == "random" else [0]):
        tr, te = daily_split(days, kind, seed)
        y_tr, y_te, p_te = aqi[days][tr], aqi[days][te], pers[days][te]
        exp2.append({"split": kind, "seed": seed, "setup": "Persistence (today = tomorrow)",
                     **score_daily(y_te, p_te, p_te)})
        for name, X in SETS.items():
            Xd = X.reindex(days)
            mdl = make_pipeline(SimpleImputer(strategy="median"),
                                RandomForestRegressor(n_estimators=300, min_samples_leaf=5,
                                                      n_jobs=-1, random_state=seed))
            mdl.fit(Xd[tr], y_tr)
            exp2.append({"split": kind, "seed": seed, "setup": name,
                         **score_daily(y_te, mdl.predict(Xd[te]), p_te)})

exp2 = pd.DataFrame(exp2)
METRICS = ["R2", "MAE", "exact_acc", "within_1", "skill_vs_pers"]
print("Evaluation days:", len(days), "| chronological test days:", daily_split(days, "chronological", 0)[1].sum())
exp2_mean = exp2.groupby(["setup", "split"])[METRICS].mean().round(3).unstack("split")
exp2_std  = exp2[exp2.split == "random"].groupby("setup")[METRICS].std().round(3)
print("\nMEAN\n", exp2_mean)
print("\nSTD over random seeds\n", exp2_std)

Evaluation days: 756 | chronological test days: 344

MEAN
                                                R2                  MAE  \
split                               chronological random chronological   
setup                                                                    
A. same-day pollutants (literature)         0.991  0.996         3.800   
B. weather only                             0.741  0.767        27.544   
C. day-ahead forecast                       0.862  0.835        20.989   
Persistence (today = tomorrow)              0.840  0.777        21.441   

                                                exact_acc         \
split                                random chronological random   
setup                                                              
A. same-day pollutants (literature)   2.309         0.942  0.977   
B. weather only                      24.889         0.494  0.643   
C. day-ahead forecast                21.359         0.648  0.684   
Persistence (t

In [12]:
# ===================== 12. SAVE RESULTS =====================
qc_report.to_csv(f"{RESULTS}/qc_report.csv")
tab_B.to_csv(f"{RESULTS}/goalB_pollutants.csv")
tab_A.to_csv(f"{RESULTS}/goalA_pollutants.csv")
season_tab.to_csv(f"{RESULTS}/goalB_aqi_by_season.csv")
aqi_cmp.to_csv(f"{RESULTS}/aqi_persistence_vs_B_vs_A.csv")
exp1.to_csv(f"{RESULTS}/exp1_random_vs_chrono_raw.csv", index=False)
r2_wide.to_csv(f"{RESULTS}/exp1_r2_inflation.csv")
exp2.to_csv(f"{RESULTS}/exp2_literature_setups_raw.csv", index=False)
exp2_mean.to_csv(f"{RESULTS}/exp2_literature_setups_mean.csv")
print("Saved to", os.path.abspath(RESULTS), ":", sorted(os.listdir(RESULTS)))

Saved to /Users/ashwanishah/Desktop/AQI/results : ['aqi_persistence_vs_B_vs_A.csv', 'exp1_r2_inflation.csv', 'exp1_random_vs_chrono_raw.csv', 'exp2_literature_setups_mean.csv', 'exp2_literature_setups_raw.csv', 'goalA_pollutants.csv', 'goalB_aqi_by_season.csv', 'goalB_pollutants.csv', 'qc_report.csv']


In [14]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score

# ============ 0. SETTINGS ============
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
SPLIT_DATE = "2025-01-01"          # train before this date
TEST_END   = "2026-01-01"          # test = 2025
AQI_POLS   = ["PM2.5", "PM10", "NO2", "SO2", "OZONE", "NH3"]   # CO left out (faulty sensor at Patia)
WEATHER    = ["AT", "RH", "SR"]    # add "wind_speed_10m", "boundary_layer_height" if you have them

# ============ 1. DAILY CONCENTRATIONS (CPCB averaging rules) ============
daily = pd.DataFrame(index=data["PM2.5"].resample("D").mean().index)
for p in ["PM2.5", "PM10", "NO2", "SO2", "NH3"]:
    day_mean  = data[p].resample("D").mean()
    day_count = data[p].resample("D").count()
    daily[p]  = day_mean.where(day_count >= 16)          # 24-h mean, needs at least 16 hours

daily["OZONE"] = data["OZONE"].rolling("8h", min_periods=6).mean().resample("D").max()   # max 8-h mean

# ============ 2. DAILY AQI AND CATEGORY (CPCB formula) ============
BREAKPOINTS = {
    "PM2.5": [0, 30, 60, 90, 120, 250, 380],
    "PM10":  [0, 50, 100, 250, 350, 430, 510],
    "NO2":   [0, 40, 80, 180, 280, 400, 520],
    "OZONE": [0, 50, 100, 168, 208, 748, 1000],
    "SO2":   [0, 40, 80, 380, 800, 1600, 2400],
    "NH3":   [0, 200, 400, 800, 1200, 1800, 2400],
}
INDEX  = [0, 50, 100, 200, 300, 400, 500]
LABELS = ["Good", "Satisfactory", "Moderate", "Poor", "Very Poor", "Severe"]

sub_index = pd.DataFrame(index=daily.index)
for p in AQI_POLS:
    sub_index[p] = np.interp(daily[p], BREAKPOINTS[p], INDEX)
    sub_index.loc[daily[p].isna(), p] = np.nan

enough = (sub_index.notna().sum(axis=1) >= 3) & (sub_index["PM2.5"].notna() | sub_index["PM10"].notna())
daily["AQI"] = sub_index.max(axis=1).where(enough)
daily["category"] = pd.cut(daily["AQI"], bins=[-1, 50, 100, 200, 300, 400, 10000],
                           labels=LABELS).astype(object)

# ============ 3. INPUT SETS ============
# (a) same-day pollutants  -> what the literature does (Pande, Gupta, Mubeen, Nallakaruppan)
same_day = daily[AQI_POLS].copy()

# (b) weather only (same day) + calendar
weather = data[WEATHER].resample("D").mean()
weather["RF_sum"]    = data["RF"].resample("D").sum(min_count=1)
weather["WD_sin"]    = np.sin(np.deg2rad(data["WD"])).resample("D").mean()
weather["WD_cos"]    = np.cos(np.deg2rad(data["WD"])).resample("D").mean()
weather["month_sin"] = np.sin(2 * np.pi * weather.index.month / 12)
weather["month_cos"] = np.cos(2 * np.pi * weather.index.month / 12)
weather["dow"]       = weather.index.dayofweek

# (c) day-ahead: today's weather + YESTERDAY's pollutants and AQI (a real forecast)
yesterday = daily[AQI_POLS + ["AQI"]].shift(1).add_suffix("_yesterday")
day_ahead = pd.concat([weather, yesterday], axis=1)

FEATURE_SETS = {
    "Same-day pollutants (literature setup)": same_day,
    "Weather only": weather,
    "Day-ahead (weather + yesterday)": day_ahead,
}

# ============ 4. TARGET, PERSISTENCE, EVALUATION DAYS ============
y = daily["category"]
persistence = y.shift(1)                       # tomorrow's category = today's category

days = y.index[y.notna() & persistence.notna() & (y.index < TEST_END)]
y, persistence = y[days], persistence[days]
print("Days available:", len(days))

# ============ 5. HELPERS ============
def make_split(days, kind, seed):
    if kind == "chronological":
        return days[days < SPLIT_DATE], days[days >= SPLIT_DATE]
    train_days, test_days = train_test_split(days, test_size=0.2, random_state=seed)
    return train_days, test_days

def evaluate(y_true, y_pred, y_pers):
    acc      = accuracy_score(y_true, y_pred)
    acc_pers = accuracy_score(y_true, y_pers)
    return {
        "accuracy":             acc,
        "macro_F1":             f1_score(y_true, y_pred, average="macro", zero_division=0),
        "Heidke_skill":         cohen_kappa_score(y_true, y_pred),          # accuracy corrected for chance
        "skill_vs_persistence": (acc - acc_pers) / (1 - acc_pers),         # 0 = same as persistence
    }

# ============ 6. RUN ============
results = []
for split in ["random", "chronological"]:
    seeds = [0, 1, 2, 3, 4] if split == "random" else [0]

    for seed in seeds:
        train_days, test_days = make_split(days, split, seed)
        y_train, y_test = y[train_days], y[test_days]
        pers_test = persistence[test_days]

        def add(name, prediction):
            results.append({"setup": name, "split": split, "seed": seed,
                            **evaluate(y_test, prediction, pers_test)})

        # --- baselines without machine learning ---
        add("CPCB formula on same-day pollutants", y_test)        # the formula IS the label -> 100% by definition
        add("Persistence (today = yesterday)", pers_test)
        add("Majority class (always most common)",
            pd.Series(y_train.mode()[0], index=test_days))

        # --- random forest classifiers ---
        for name, X in FEATURE_SETS.items():
            model = make_pipeline(
                SimpleImputer(strategy="median"),
                RandomForestClassifier(n_estimators=300, min_samples_leaf=3,
                                       random_state=seed, n_jobs=-1))
            model.fit(X.loc[train_days], y_train)
            add(name, model.predict(X.loc[test_days]))

# ============ 7. RESULTS ============
results = pd.DataFrame(results)
summary = results.groupby(["setup", "split"])[["accuracy", "macro_F1", "Heidke_skill",
                                                "skill_vs_persistence"]].mean().round(3)
print(summary.unstack("split"))

print("\nTest days (chronological):", (days >= SPLIT_DATE).sum())
print("Class counts in 2025:\n", y[days >= SPLIT_DATE].value_counts())

Days available: 756
                                            accuracy             macro_F1         Heidke_skill        skill_vs_persistence       
split                                  chronological random chronological random chronological random        chronological random
setup                                                                                                                            
CPCB formula on same-day pollutants            1.000  1.000         1.000  1.000         1.000  1.000                1.000  1.000
Day-ahead (weather + yesterday)                0.733  0.675         0.687  0.603         0.637  0.569                0.132  0.090
Majority class (always most common)            0.233  0.258         0.075  0.082         0.000  0.000               -1.491 -1.087
Persistence (today = yesterday)                0.692  0.639         0.613  0.597         0.582  0.523                0.000  0.000
Same-day pollutants (literature setup)         0.983  0.974         0.

In [16]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

# ============ 6. RUN ============
results = []
for split in ["random", "chronological"]:
    seeds = [0, 1, 2, 3, 4] if split == "random" else [0]

    for seed in seeds:
        train_days, test_days = make_split(days, split, seed)
        y_train, y_test = y[train_days], y[test_days]
        pers_test = persistence[test_days]

        def add(name, prediction):
            results.append({"setup": name, "split": split, "seed": seed,
                            **evaluate(y_test, prediction, pers_test)})

        # --- baselines without machine learning ---
        add("CPCB formula on same-day pollutants", y_test)        # the formula IS the label -> 100% by definition
        add("Persistence (today = yesterday)", pers_test)
        add("Majority class (always most common)",
            pd.Series(y_train.mode()[0], index=test_days))

        # --- random forest classifiers ---
        for name, X in FEATURE_SETS.items():
            model = make_pipeline(
                SimpleImputer(strategy="median"),
                ExtraTreesClassifier())
            model.fit(X.loc[train_days], y_train)
            add(name, model.predict(X.loc[test_days]))

# ============ 7. RESULTS ============
results = pd.DataFrame(results)
summary = results.groupby(["setup", "split"])[["accuracy", "macro_F1", "Heidke_skill",
                                                "skill_vs_persistence"]].mean().round(3)
print(summary.unstack("split"))

print("\nTest days (chronological):", (days >= SPLIT_DATE).sum())
print("Class counts in 2025:\n", y[days >= SPLIT_DATE].value_counts())

                                            accuracy             macro_F1         Heidke_skill        skill_vs_persistence       
split                                  chronological random chronological random chronological random        chronological random
setup                                                                                                                            
CPCB formula on same-day pollutants            1.000  1.000         1.000  1.000         1.000  1.000                1.000  1.000
Day-ahead (weather + yesterday)                0.721  0.659         0.655  0.571         0.620  0.547                0.094  0.051
Majority class (always most common)            0.233  0.258         0.075  0.082         0.000  0.000               -1.491 -1.087
Persistence (today = yesterday)                0.692  0.639         0.613  0.597         0.582  0.523                0.000  0.000
Same-day pollutants (literature setup)         0.907  0.933         0.895  0.897         0